[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hramdasan/elixir_summerschool_26/blob/main/answers/5-transformers.ipynb)

# 5 Transformers for protein sequences

In this notebook, we will download a pretrained protein transformer, inspect its tokenizer, predict a masked amino acid, and visualize its attention weights. We will then extract protein embeddings and use them in a classification task.

We use [ESM-2](https://huggingface.co/facebook/esm2_t6_8M_UR50D), a model pretrained using masked-residue prediction. The selected version has six transformer layers and approximately eight million parameters. It is small enough to run on a CPU.

## Before you start: Colab setup

In Colab, select **File → Save a copy in Drive** to keep your answers. Run the setup cell below first, then run the remaining cells from top to bottom with **Shift + Enter**. This is provided setup code; you do not need to modify it for the exercises.

The cell installs the packages needed by this notebook. If Colab asks you to restart the session after installation, restart it and run from the setup cell again. A standard CPU runtime is sufficient.

In local Jupyter, this cell does not install anything. Follow the [README installation instructions](https://github.com/hramdasan/elixir_summerschool_26#run-locally) instead.

The model weights and protein CSV download automatically when their sections run. No manual file upload is needed.

The setup also enables interactive widgets in Colab. Static plots remain available if your notebook viewer cannot display the controls.

In [ ]:
# Provided setup: run once before the exercises in a fresh Colab session.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q "torch>=2.6,<3" "numpy>=2,<3" "pandas>=2.2,<3" "scikit-learn>=1.6,<2" "matplotlib>=3.9,<4" "ipywidgets>=8,<9" "transformers==4.57.6"
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Setup complete. Continue with the cells below.")
else:
    print("Local Jupyter: use the README installation instructions, then continue below.")

## 5.1 Import packages and choose the device

Run the setup cell above before these imports in Colab. For local Jupyter, install the packages listed in `requirements.txt` as described in the README.

The first model-loading cell downloads the weights. Later runs reuse them from `model_cache/`. The protein CSV downloads automatically in section 5.7 if it is not already available locally. This notebook calculates the model outputs itself; it does not load precomputed embeddings.

In [1]:
# %pip install "torch>=2.6,<3" "transformers==4.57.6" "numpy>=2,<3" "pandas>=2.2,<3" "scikit-learn>=1.6,<2" "matplotlib>=3.9,<4" "ipywidgets>=8,<9"

from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 5.2 Downloading and loading ESM-2

The tokenizer converts amino-acid letters into token IDs. The model converts those tokens into contextual representations and scores for predicting masked residues.

We load a fixed model revision so that the weights do not change between runs. `eval()` disables training-time dropout. We also request `attn_implementation="eager"` so the model returns the explicit attention matrices needed for the visualization; some optimized attention implementations do not return these weights.

In [2]:
MODEL_ID = "facebook/esm2_t6_8M_UR50D"
REVISION = "c731040fcd8d73dceaa04b0a8e6329b345b0f5df"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, cache_dir="model_cache")
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID, revision=REVISION, cache_dir="model_cache", attn_implementation="eager"
).to(device).eval()

print("Layers:", model.config.num_hidden_layers)
print("Attention heads per layer:", model.config.num_attention_heads)
print("Features per residue:", model.config.hidden_size)
print("Position representation:", model.config.position_embedding_type)
print("Parameters:", sum(p.numel() for p in model.parameters()))

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

c:\Users\harik\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harik\Desktop\Doctoral_Project\Elixir\ai-ml-summer-school\ai-ml-summer-school\answers\model_cache\models--facebook--esm2_t6_8M_UR50D. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layers: 6
Attention heads per layer: 20
Features per residue: 320
Position representation: rotary
Parameters: 7512474


## 5.3 Inspecting tokens

We start with a short illustrative peptide. You can replace it with another amino-acid sequence. ESM-2 usually represents each amino acid with one token and adds special tokens around the sequence.

Token IDs are lookup addresses, not continuous measurements. A larger ID does not mean a larger or more important amino acid. `<cls>` and `<eos>` are special tokens at the start and end; they are not residues.

In [3]:
sequence = "MKTIIALSYIFCLVFADYKDDDDK"
encoded = tokenizer(sequence, return_tensors="pt", return_special_tokens_mask=True)
token_strings = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0].tolist())

token_table = pd.DataFrame({
    "token_position": np.arange(len(token_strings)),
    "token": token_strings,
    "token_id": encoded["input_ids"][0].tolist(),
    "is_special": encoded["special_tokens_mask"][0].bool().tolist(),
})
display(token_table)
print("Residues:", len(sequence), "Tokens:", len(token_strings))
print("Special tokens:", tokenizer.special_tokens_map)

,token_position,token,token_id,is_special
0,0,<cls>,0,True
1,1,M,20,False
2,2,K,15,False
3,3,T,11,False
4,4,I,12,False
5,5,I,12,False
6,6,A,5,False
7,7,L,4,False
8,8,S,8,False
9,9,Y,19,False


Residues: 24 Tokens: 26
Special tokens: {'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'cls_token': '<cls>', 'mask_token': '<mask>'}


*Assignment:* Replace the peptide with `ACDEFGHIKLMNPQRSTVWY`. How many residues and tokens are there? Do repeated amino acids get the same token ID?

In [ ]:
example_sequence = "ACDEFGHIKLMNPQRSTVWY"
example_ids = tokenizer(example_sequence)["input_ids"]
print(tokenizer.convert_ids_to_tokens(example_ids))
print("Residues:", len(example_sequence), "Tokens:", len(example_ids))
print("Repeated A:", tokenizer("AAA")["input_ids"])

*Answer:* There are 20 residues and 22 tokens, including the two special tokens. Repeated residues have the same token ID. Their later contextual representations can differ because their positions and neighboring residues differ.

### 5.3.1 Padding sequences into a batch

Different lengths need padding to form a rectangular batch. ESM-2 uses a separate `<pad>` token. `attention_mask` is 1 for input tokens to retain and 0 for padding. This differs from the Boolean key-padding mask used by PyTorch's encoder in the previous notebook, where True means ignore.

The start and end tokens are valid model inputs, so their attention-mask entries are 1. We exclude them separately when pooling residue representations.

In [6]:
batch = tokenizer([sequence, "MKWVTFIS"], padding=True, return_tensors="pt", return_special_tokens_mask=True)
for row in range(2):
    print("Tokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][row].tolist()))
    print("Attention mask:", batch["attention_mask"][row].tolist())
print("Batch shape:", batch["input_ids"].shape)

Tokens: ['<cls>', 'M', 'K', 'T', 'I', 'I', 'A', 'L', 'S', 'Y', 'I', 'F', 'C', 'L', 'V', 'F', 'A', 'D', 'Y', 'K', 'D', 'D', 'D', 'D', 'K', '<eos>']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Tokens: ['<cls>', 'M', 'K', 'W', 'V', 'T', 'F', 'I', 'S', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Batch shape: torch.Size([2, 26])


## 5.4 Predicting a masked amino acid

Pretraining asked the model to predict hidden residues from the surrounding sequence. We can try the same operation by replacing one amino acid with `<mask>`.

The model outputs one score per vocabulary token at each position. Softmax converts the scores at the masked position into a probability distribution. These probabilities describe the model's sequence predictions, not the probability of a biological function or a mutation being safe.

In [4]:
def predict_mask(sequence, residue_position, top_k=5):
    # residue_position counts the amino acids from zero, before special tokens are added.
    if not 0 <= residue_position < len(sequence):
        raise ValueError("Choose a residue position within the sequence.")
    masked_sequence = sequence[:residue_position] + tokenizer.mask_token + sequence[residue_position + 1:]
    inputs = tokenizer(masked_sequence, return_tensors="pt").to(device)
    mask_position = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    with torch.inference_mode():
        output = model(**inputs)
    probabilities = torch.softmax(output.logits[0, mask_position], dim=-1)
    # Display the top standard amino acids, retaining their original vocabulary probabilities.
    amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
    amino_acid_ids = tokenizer.convert_tokens_to_ids(amino_acids)
    table = pd.DataFrame({"amino_acid": amino_acids,
                          "probability": probabilities[amino_acid_ids].cpu().numpy()})
    print("Masked input:", masked_sequence)
    print("Original residue:", sequence[residue_position])
    return table.sort_values("probability", ascending=False).head(top_k)

display(predict_mask(sequence, residue_position=10))

Masked input: MKTIIALSYI<mask>CLVFADYKDDDDK
Original residue: F


,amino_acid,probability
9,L,0.222767
7,I,0.159362
0,A,0.139730
17,V,0.120811
4,F,0.106459


*Assignment:* Change the masked position and compare the predictions. Is the original residue always the highest-scoring one? Try a sequence of your own.

In [7]:
for residue_position in [0, 5, 15]:
    print("Residue position:", residue_position)
    display(predict_mask(sequence, residue_position))

Residue position: 0
Masked input: <mask>KTIIALSYIFCLVFADYKDDDDK
Original residue: M


,amino_acid,probability
10,M,0.952171
8,K,0.007113
9,L,0.006679
7,I,0.005530
4,F,0.004266


Residue position: 5
Masked input: MKTII<mask>LSYIFCLVFADYKDDDDK
Original residue: A


,amino_acid,probability
9,L,0.341690
7,I,0.278862
4,F,0.138819
17,V,0.083326
0,A,0.029046


Residue position: 15
Masked input: MKTIIALSYIFCLVF<mask>DYKDDDDK
Original residue: A


,amino_acid,probability
1,C,0.137086
0,A,0.116962
15,S,0.092896
9,L,0.086557
4,F,0.074196


*Answer:* The original residue need not rank first. The model gives plausible residues based on its learned sequence statistics and this context. A short illustrative peptide is not sufficient evidence for a functional or mutational conclusion.

## 5.5 Extracting attention weights

We now run the **unmasked** peptide through the model. Setting `output_attentions=True` returns the attention weights for every layer and head.

There is one tensor per layer, with shape `(batch, heads, query positions, key positions)`. Each row describes how one query position combines information from key/value positions. The model uses bidirectional attention, so a position can read both earlier and later positions.

ESM-2 uses rotary positional information. Unlike the toy model's added position embeddings, rotary positions affect the query/key calculation inside attention.

In [ ]:
# Use a short sequence so every token label is readable in the plot.
attention_sequence = sequence
if len(attention_sequence) > 60:
    raise ValueError("Use at most 60 residues for this visualization; shorter sequences give readable plots.")
attention_inputs = tokenizer(attention_sequence, return_tensors="pt").to(device)
attention_tokens = tokenizer.convert_ids_to_tokens(attention_inputs["input_ids"][0].cpu().tolist())

with torch.inference_mode():
    attention_output = model(**attention_inputs, output_attentions=True, output_hidden_states=True)

# Stack layer tensors after removing the one-example batch axis.
attention = torch.stack([layer[0].cpu() for layer in attention_output.attentions])
print("Attention shape (layers, heads, queries, keys):", tuple(attention.shape))
print("Row sums:", attention[0, 0].sum(dim=-1))
assert torch.allclose(attention.sum(dim=-1), torch.ones_like(attention.sum(dim=-1)), atol=1e-5)

### 5.5.1 Visualizing one head

In the heatmap, rows are query positions and columns are key positions. Darker squares have larger weights. The red horizontal line marks the selected query; the bar chart shows that same row in more detail.

All tokens, including `<cls>` and `<eos>`, are shown so each row retains its complete probability mass. Token index 0 is `<cls>`; the first residue is at token index 1. The color scale is fixed from 0 to 1 so different heads remain comparable.

A diagonal pattern means positions attend strongly to themselves or their neighbors. Attention to a distant position means the model draws on that position in this layer and head. Attention weights alone do not establish physical contacts, causality or biological importance.

In [ ]:
def plot_attention(layer=0, head=0, query=1):
    weights = attention[layer, head].numpy()
    labels = [f"{i}:{token}" for i, token in enumerate(attention_tokens)]
    n = len(labels)
    fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1]})
    image = ax_map.imshow(weights, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    step = max(1, math.ceil(n / 28))
    ticks = np.arange(0, n, step)
    ax_map.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=8)
    ax_map.set_yticks(ticks, [labels[i] for i in ticks], fontsize=8)
    ax_map.axhline(query, color="crimson", linewidth=1)
    ax_map.set(xlabel="Key position (supplies information)", ylabel="Query position (receives information)",
               title=f"Layer {layer + 1}, head {head + 1}")
    fig.colorbar(image, ax=ax_map, fraction=0.046, label="Attention weight")
    ax_bar.bar(np.arange(n), weights[query], color="#277DA1")
    ax_bar.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=8)
    ax_bar.set(xlabel="Key position", ylabel="Attention weight", ylim=(0, 1),
               title=f"Query {labels[query]}: one row of the heatmap")
    plt.tight_layout()
    plt.show()

plot_attention(layer=0, head=0, query=1)

### 5.5.2 Comparing layers, heads and query positions

The controls below change the view without rerunning the transformer. Layers and heads are displayed starting at 1; the underlying tensors use zero-based indexing. The query menu uses the token indices printed in the table.

If your notebook viewer cannot display widgets, change the arguments to `plot_attention` in the previous cell. The static heatmap and bar chart work without widgets.

In [ ]:
import ipywidgets as widgets

layer_control = widgets.Dropdown(options=[(str(i + 1), i) for i in range(attention.shape[0])],
                                 value=0, description="Layer:")
head_control = widgets.Dropdown(options=[(str(i + 1), i) for i in range(attention.shape[1])],
                                value=0, description="Head:")
query_control = widgets.Dropdown(options=[(f"{i}: {token}", i) for i, token in enumerate(attention_tokens)],
                                 value=1, description="Query:")
attention_view = widgets.interactive_output(plot_attention,
    {"layer": layer_control, "head": head_control, "query": query_control})
display(widgets.HBox([layer_control, head_control, query_control]), attention_view)

*Assignment:* Compare an early layer with a later layer and at least two heads. Find a head that emphasizes nearby positions, special tokens or a broader set of positions. Do all heads show the same pattern?

In [ ]:
plot_attention(layer=0, head=1, query=5)
plot_attention(layer=5, head=1, query=5)

*Answer:* Different heads and layers can distribute attention differently; describe the actual plots rather than assuming a specific head has a fixed biological role. The observed pattern depends on the sequence. A large weight is one part of a multi-head computation and should not be treated as a standalone explanation of the prediction.

## 5.6 From residue representations to a protein vector

The model also returns hidden representations. There is one representation for every token. We can average the final-layer **residue** vectors to obtain one vector per protein, excluding padding and special tokens.

This function uses the encoder inside the masked-language model. It does not need masked-token prediction scores or attention matrices, so those outputs are disabled here to save memory. We keep the transformer frozen.

In [ ]:
def embed_sequences(sequences, batch_size=4):
    vectors = []
    for start in range(0, len(sequences), batch_size):
        inputs = tokenizer(list(sequences[start:start + batch_size]), padding=True,
                           return_tensors="pt", return_special_tokens_mask=True)
        special_mask = inputs.pop("special_tokens_mask").to(device)
        inputs = inputs.to(device)
        residue_mask = inputs["attention_mask"].bool() & ~special_mask.bool()
        with torch.inference_mode():
            hidden = model.esm(**inputs, output_attentions=False, output_hidden_states=False).last_hidden_state
            pooled = (hidden * residue_mask.unsqueeze(-1)).sum(1) / residue_mask.sum(1, keepdim=True)
        vectors.append(pooled.cpu().numpy())
    return np.concatenate(vectors).astype(np.float32)

example_vectors = embed_sequences([sequence, "MKWVTFIS"])
print("Embedding shape:", example_vectors.shape)
print("First eight values:", np.round(example_vectors[0, :8], 3))
# Padding should not change a sequence's pooled vector.
alone = embed_sequences(["MKWVTFIS"])
assert np.allclose(alone[0], example_vectors[1], atol=1e-4, rtol=1e-4)

## 5.7 Loading a protein dataset

The accompanying CSV contains 120 reviewed UniProt entries from E. coli K-12: 60 annotated as cytoplasmic and 60 as membrane-associated. Labels are **0 = cytoplasm, 1 = membrane**. Only sequence-derived features are used as model inputs.

The data have predefined training, validation and test partitions. Detected similar sequences were grouped before splitting; each group appears in only one partition. The annotations and selection procedure are described in `data/README.md`.

This is a small teaching dataset. Membrane annotation includes different kinds of membrane association, residual homology may remain, and ESM-2 may have seen these sequences or their relatives during pretraining. The results therefore do not establish generalization to unseen protein families.

In [ ]:
# This supports opening the student notebook or the matching answer notebook.
data_path = next((p for p in [Path("data/proteins.csv"), Path("../data/proteins.csv")]
                  if p.is_file()), None)
if data_path is None:
    from urllib.request import urlopen

    # A fixed repository revision keeps the teaching dataset reproducible.
    data_url = "https://raw.githubusercontent.com/hramdasan/elixir_summerschool_26/57c6a252e305256756949986951b66a2cc1f3200/data/proteins.csv"
    data_path = Path("data/proteins.csv")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(data_url, timeout=30) as response:
        csv_bytes = response.read()
    data_path.write_bytes(csv_bytes)
    print("Downloaded proteins.csv into the data folder.")
proteins = pd.read_csv(data_path)
display(proteins[["accession", "location", "length", "split"]].head())
display(pd.crosstab(proteins.split, proteins.location))
assert proteins.groupby("group").split.nunique().max() == 1

# Calculate the embeddings now using the downloaded model.
protein_vectors = embed_sequences(proteins.sequence.tolist())
print("Calculated vectors:", protein_vectors.shape)

## 5.8 Comparing embeddings with amino-acid composition

Amino-acid composition gives one fraction for each of the 20 standard residues. It ignores order but may already be useful for membrane association. We will compare it with the 320-dimensional ESM-2 vectors using the same logistic-regression classifier family.

Scaling is fitted on the training partition only. The regularization parameter `C` is selected using validation ROC-AUC; smaller C means stronger regularization. We do not use the test data to select it.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay

def amino_acid_composition(sequences):
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    return np.array([[seq.count(aa) / len(seq) for aa in amino_acids] for seq in sequences], dtype=np.float32)

features = {"Amino-acid composition": amino_acid_composition(proteins.sequence),
            "ESM-2 embeddings": protein_vectors}
train = proteins.split.eq("train").to_numpy()
val = proteins.split.eq("val").to_numpy()
test = proteins.split.eq("test").to_numpy()
y = proteins.label.to_numpy()

*Assignment:* Compare the two representations. Add the other C values and select each representation’s classifier using validation. Does the transformer representation improve the validation score?

In [ ]:
candidate_C = [0.01, 0.1, 1.0]
selected = {}
validation_results = []
for name, x in features.items():
    best_score = -np.inf
    for C in candidate_C:
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=42))
        classifier.fit(x[train], y[train])
        score = roc_auc_score(y[val], classifier.predict_proba(x[val])[:, 1])
        validation_results.append({"representation": name, "C": C, "validation_ROC_AUC": score})
        if score > best_score:
            selected[name] = classifier
            best_score = score
display(pd.DataFrame(validation_results))

*Answer:* Select C separately for each representation on validation. Ties retain the first setting. It is acceptable for a simple baseline to perform well; embeddings are not guaranteed to win. There are only 24 validation examples, so small differences should not be overinterpreted.

## 5.9 Evaluating the selected classifiers

Each classifier has now been selected without consulting the test scores. Evaluate both methods on the same held-out proteins. The threshold of 0.5 is fixed before evaluation.

There are only 24 test proteins, so one error changes accuracy by about four percentage points. Interpret the confusion matrices alongside the scores.

In [ ]:
test_results = []
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, x) in zip(axes, features.items()):
    probability = selected[name].predict_proba(x[test])[:, 1]
    predicted = probability >= 0.5
    test_results.append({"representation": name,
                         "accuracy": accuracy_score(y[test], predicted),
                         "balanced_accuracy": balanced_accuracy_score(y[test], predicted),
                         "ROC_AUC": roc_auc_score(y[test], probability)})
    ConfusionMatrixDisplay.from_predictions(y[test], predicted,
        display_labels=["cytoplasm", "membrane"], cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()
display(pd.DataFrame(test_results).set_index("representation").round(3))

*Question:* Which parameters were updated in the classification task? What additional work would be needed to fine-tune the transformer? What can this small test set tell us about proteins from another organism?

*Answer:* Only the logistic-regression parameters were fitted. Fine-tuning would require gradients through the transformer, an appropriate training procedure and further validation. The small within-organism test cannot establish performance on another organism, particularly given potential pretraining overlap and remaining related sequences.